In [1]:
import numpy as np
import matplotlib.pyplot as plt
import stim
import pymatching
import sinter
from typing import List
import time
import pandas as pd
from IPython.display import HTML
import lmfit
import itertools
from lmfit import Model
from stimbposd import SinterDecoder_BPOSD, sinter_decoders
from stimbposd import BPOSD
from itertools import combinations
import galois
import ldpc
from ldpc import bposd_decoder

In [101]:
##Non Periodic boundary conditions

h=4   ###Height of grid
w=55  ###Width of grid, make odd k=27-6
H=[]
stabs=[]
packed=[]
for i in range(1,h):
    for j in range(0,w):          
        stab=[]

        stab.append([[i,j], [i-1,j-1],[i-1,j],[i-1,j+1]])

        sta=np.zeros([h,w])
        for pack in stab[0]:
            if pack[1] >= 0 and pack[1] < w:
                sta[pack[0]][pack[1]]=1
        sta=sta[::-1]
        final_sta=[sta[0],sta[1][1:-1],sta[2][2:-2],sta[3][3:-3]]
        final_sta = np.concatenate(final_sta).tolist()
        if np.sum(final_sta) > 0:
            H.append(final_sta)

        
H=np.array(H).astype(int)




In [108]:
def gf2_rref(A):
    A = A.copy()
    m, n = A.shape
    row = 0
    pivots = []

    for col in range(n):
        pivot = None
        for r in range(row, m):
            if A[r, col] == 1:
                pivot = r
                break
        if pivot is None:
            continue

        A[[row, pivot]] = A[[pivot, row]]
        pivots.append(col)

        for r in range(m):
            if r != row and A[r, col] == 1:
                A[r] ^= A[row]

        row += 1
        if row == m:
            break

    return A, pivots

def nullspace_gf2(H):
    R, pivots = gf2_rref(H)
    n = H.shape[1]
    free = [i for i in range(n) if i not in pivots]

    G = []
    for f in free:
        v = np.zeros(n, dtype=int)
        v[f] = 1
        for i, p in enumerate(pivots):
            v[p] = R[i, f]
        G.append(v)

    return np.array(G)

G_LDPC = nullspace_gf2(H)   # 12 × 23

# -----------------------------
# Convert G to systematic form
# -----------------------------
def systematic_form(G):
    G = G.copy()
    rows, cols = G.shape
    r = 0
    col_perm = list(range(cols))

    for c in range(cols):
        if r >= rows:
            break
        for rr in range(r, rows):
            if G[rr, c]:
                G[[r, rr]] = G[[rr, r]]
                col_perm[r], col_perm[c] = col_perm[c], col_perm[r]
                G[:, [r, c]] = G[:, [c, r]]
                for rrr in range(rows):
                    if rrr != r and G[rrr, r]:
                        G[rrr] ^= G[r]
                r += 1
                break

    return G, col_perm

#G_sys, perm = systematic_form(G)

print(len(G_LDPC))



49


In [103]:
code_supports = []
for j in range(G_LDPC.shape[1]):      # loop over columns
    rows = np.where(G_LDPC[:, j] == 1)[0]
    code_supports.append([int(r % 7) for r in rows])

print(code_supports)

[[0], [0, 1], [1, 2], [0, 2, 3], [1, 3, 4], [0, 2, 4, 5], [0, 1, 3, 5, 6], [1, 2, 4, 6, 0], [2, 3, 5, 0, 1], [3, 4, 6, 1, 2], [4, 5, 0, 2, 3], [5, 6, 1, 3, 4], [6, 0, 2, 4, 5], [0, 1, 3, 5, 6], [1, 2, 4, 6, 0], [2, 3, 5, 0, 1], [3, 4, 6, 1, 2], [4, 5, 0, 2, 3], [5, 6, 1, 3, 4], [6, 0, 2, 4, 5], [0, 1, 3, 5, 6], [1, 2, 4, 6, 0], [2, 3, 5, 0, 1], [3, 4, 6, 1, 2], [4, 5, 0, 2, 3], [5, 6, 1, 3, 4], [6, 0, 2, 4, 5], [0, 1, 3, 5, 6], [1, 2, 4, 6, 0], [2, 3, 5, 0, 1], [3, 4, 6, 1, 2], [4, 5, 0, 2, 3], [5, 6, 1, 3, 4], [6, 0, 2, 4, 5], [0, 1, 3, 5, 6], [1, 2, 4, 6, 0], [2, 3, 5, 0, 1], [3, 4, 6, 1, 2], [4, 5, 0, 2, 3], [5, 6, 1, 3, 4], [6, 0, 2, 4, 5], [0, 1, 3, 5, 6], [1, 2, 4, 6, 0], [2, 3, 5, 0, 1], [3, 4, 6, 1, 2], [4, 5, 0, 2, 3], [5, 6, 1, 3, 4], [6, 0, 2, 4, 5], [0, 1, 3, 5, 6], [1, 2, 4, 6], [2, 3, 5], [3, 4, 6], [4, 5], [5, 6], [6], [0], [1], [0, 2], [1, 3], [0, 2, 4], [1, 3, 5], [2, 4, 6], [3, 5, 0], [4, 6, 1], [5, 0, 2], [6, 1, 3], [0, 2, 4], [1, 3, 5], [2, 4, 6], [3, 5, 0], [4, 6, 

In [84]:
from collections import Counter

def uniqueness(lst):
    if len(lst) == 0:      #  allow empty lists
        return True
    
    counts = Counter(lst)
    has_unique = any(count == 1 for count in counts.values())
    return has_unique


for i in range(len(code_supports)):
    ans = uniqueness(code_supports[i])
    if ans == False:
        print("Failure at qubit", i)

In [85]:
from collections import Counter

def pairs_dominate(lst):
    if len(lst) == 0:     # allow empty lists
        return False      # not a failure
    
    counts = Counter(lst)
    
    singles = sum(1 for c in counts.values() if c == 1)
    pairs   = sum(1 for c in counts.values() if c == (2 or 3 or 4))
    
    return pairs >= singles   # True means FAILURE

for i in range(len(code_supports)):
    if pairs_dominate(code_supports[i]):
        print("Failure at qubit", i)

Testing Entirely Separable Code

In [119]:
H_outer= [
    [1, 0, 1, 0, 1, 0, 1],
    [0, 1, 1, 0, 0, 1, 1],
    [0, 0, 0, 1, 1, 1, 1]]

H_full=[]

for j in range(len(H_outer)):
    row_0 = []
    row_1 = []
    row_2 = []
    row_3 = []
    row_4 = []
    row_5 = []
    row_6 = []
    for i in H_outer[j]:
        row_0 += [int(i), 0, 0,0,0,0,0]
        row_1 += [0, int(i), 0,0,0,0,0]
        row_2 += [0, 0, int(i),0,0,0,0]
        row_3 += [0, 0, 0,int(i),0,0,0]
        row_4 += [0, 0, 0,0,int(i),0,0]
        row_5 += [0, 0, 0,0,0,int(i),0]
        row_6 += [0, 0, 0,0,0,0,int(i)]
    H_full.append(row_0)
    H_full.append(row_1)
    H_full.append(row_2)
    H_full.append(row_3)
    H_full.append(row_4)
    H_full.append(row_5)
    H_full.append(row_6)



Hx = (H_full @ G_LDPC) % 2



bp_osd = BpOsdDecoder(
            Hx,
            error_rate = 0.1,
            bp_method = 'product_sum',
            max_iter = 30,
            schedule = 'serial',
            osd_method = 'osd_cs', #set to OSD_0 for fast solve
            osd_order = 5
        )

for i in range(100):

    error = np.zeros(208)
    place=np.random.randint(0,208)
    error[place]=1
    syndrome = Hx @ error
    decoding = bp_osd.decode(syndrome)

    if np.array_equal(decoding, error):
        pass
    else:
        print("Failure")
        print(syndrome, place,code_supports[place])



Failure
[0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 1.] 158 [6]
Failure
[1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.] 159 [0]
Failure
[1. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.] 109 [0, 1]
Failure
[0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 1. 0.] 106 [5]
Failure
[0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 1.] 54 [6]


In [118]:
print(len(Hx[0]))

208


In [9]:
H_outer= [
    [1, 0, 1, 0, 1, 0, 1],
    [0, 1, 1, 0, 0, 1, 1],
    [0, 0, 0, 1, 1, 1, 1]]

H_full=[]

for j in range(len(H_outer)):
    row_0 = []
    row_1 = []
    row_2 = []
    for i in H_outer[j]:
        row_0 += [int(i), 0, 0]
        row_1 += [0, int(i), 0]
        row_2 += [0, 0, int(i)]
    H_full.append(row_0)
    H_full.append(row_1)
    H_full.append(row_2)

print(H_full)

[[1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0], [0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0], [0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1], [0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0], [0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0], [0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1], [0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 1]]


In [11]:
Hx= (H_full @ G_LDPC) % 2

print(Hx)

[[1 1 0 1 0 1 0 1 0 1 0 1 0 1 0 1 0 1 0 1 0 1 0 1 1 0 0 1 0 1 0 1 0 1 0 1
  0 1 0 1 0 1 0 1 0 1 0 1 0 1 0 0 1 1 1 0 0 0 1 1 1 0 0 0 1 1 1 0 0 0 1 1
  1 0 0 1 0 0 0 0 0 1 0 0 0 0 0 1 0 0 0 0 0 1 0 0]
 [0 1 1 0 1 0 1 0 1 0 1 0 1 0 1 0 1 0 1 0 1 0 1 0 1 1 0 0 1 0 1 0 1 0 1 0
  1 0 1 0 1 0 1 0 1 0 1 0 1 0 1 0 0 1 1 1 0 0 0 1 1 1 0 0 0 1 1 1 0 0 0 1
  1 1 0 0 1 0 0 0 0 0 1 0 0 0 0 0 1 0 0 0 0 0 1 0]
 [0 0 1 1 0 1 0 1 0 1 0 1 0 1 0 1 0 1 0 1 0 1 0 1 0 1 1 0 0 1 0 1 0 1 0 1
  0 1 0 1 0 1 0 1 0 1 0 1 0 1 0 1 0 0 1 1 1 0 0 0 1 1 1 0 0 0 1 1 1 0 0 0
  1 1 1 0 0 1 0 0 0 0 0 1 0 0 0 0 0 1 0 0 0 0 0 1]
 [0 0 0 1 1 0 0 1 1 0 0 1 1 0 0 1 1 0 0 1 1 0 0 1 1 0 0 0 0 0 1 0 1 1 1 1
  0 1 0 0 0 0 1 0 1 1 1 1 0 1 0 0 0 0 0 1 1 1 1 1 1 0 0 0 0 0 0 1 1 1 1 1
  1 0 0 0 0 0 1 0 0 1 0 0 0 0 0 0 0 0 1 0 0 1 0 0]
 [0 0 0 0 1 1 0 0 1 1 0 0 1 1 0 0 1 1 0 0 1 1 0 0 1 1 0 0 0 0 0 1 0 1 1 1
  1 0 1 0 0 0 0 1 0 1 1 1 1 0 1 0 0 0 0 0 1 1 1 1 1 1 0 0 0 0 0 0 1 1 1 1
  1 1 0 0 0 0 0 1 0 0 1 0 0 0 0 0 0 0 0 1 0 0 1 0]
 [0 0

In [35]:

import ldpc.codes
from ldpc import BpOsdDecoder


bp_osd = BpOsdDecoder(
            Hx,
            error_rate = 0.1,
            bp_method = 'product_sum',
            max_iter = 30,
            schedule = 'serial',
            osd_method = 'osd_cs', #set to OSD_0 for fast solve
            osd_order = 5
        )

for i in range(100):

    error = np.zeros(96)
    place=np.random.randint(0,96)
    error[place]=1
    syndrome = Hx @ error
    decoding = bp_osd.decode(syndrome)

    if np.array_equal(decoding, error):
        pass
    else:
        print("Failure")
        print(syndrome, place,code_supports[place])

Failure
[1. 0. 0. 0. 0. 0. 0. 0. 0.] 27 [0]
Failure
[1. 1. 0. 0. 0. 0. 0. 0. 0.] 53 [0, 1]
Failure
[1. 0. 0. 0. 0. 0. 0. 0. 0.] 75 [0]
Failure
[0. 1. 0. 1. 1. 0. 1. 1. 1.] 16 [1, 2, 1, 0, 1]
Failure
[0. 1. 0. 0. 0. 0. 0. 0. 0.] 76 [1]
Failure
[0. 1. 0. 1. 1. 0. 0. 0. 0.] 4 [1, 0, 1]
Failure
[0. 1. 0. 1. 1. 0. 1. 1. 1.] 16 [1, 2, 1, 0, 1]
Failure
[1. 0. 0. 0. 0. 0. 0. 0. 0.] 75 [0]
Failure
[1. 0. 1. 1. 0. 0. 0. 0. 0.] 7 [1, 2, 1, 0, 1]
Failure
[1. 0. 1. 0. 1. 1. 1. 0. 0.] 13 [1, 2, 1, 0, 1]
Failure
[0. 1. 0. 1. 1. 0. 0. 0. 0.] 4 [1, 0, 1]
Failure
[0. 1. 0. 1. 1. 0. 1. 1. 1.] 16 [1, 2, 1, 0, 1]
Failure
[1. 0. 0. 0. 0. 0. 0. 0. 0.] 52 [0]
Failure
[0. 1. 0. 0. 1. 0. 0. 1. 0.] 94 [1]
Failure
[1. 1. 0. 0. 0. 0. 0. 0. 0.] 53 [0, 1]
Failure
[0. 1. 0. 0. 1. 0. 0. 1. 0.] 94 [1]
Failure
[0. 1. 0. 1. 1. 0. 1. 1. 1.] 16 [1, 2, 1, 0, 1]
Failure
[0. 1. 0. 0. 0. 0. 0. 0. 0.] 76 [1]


In [38]:
error=np.zeros(96)
error[76]=1
print(Hx @ error%2)

[0. 1. 0. 0. 0. 0. 0. 0. 0.]


In [88]:
print(len(H[0]))

180


Plan is to concatenate [208,49,12] code into 7x [7,4,3] codes. Decode each one separatly and take a majority vote on the outcoming syndromes.


In [ ]:
H_outer= [
    [1, 0, 1, 0, 1, 0, 1],
    [0, 1, 1, 0, 0, 1, 1],
    [0, 0, 0, 1, 1, 1, 1]]

H_full=[]

for j in range(len(H_outer)):
    row_0 = []
    row_1 = []
    row_2 = []
    row_3 = []
    row_4 = []
    row_5 = []
    row_6 = []
    for i in H_outer[j]:
        row_0 += [int(i), 0, 0,0,0,0,0]
        row_1 += [0, int(i), 0,0,0,0,0]
        row_2 += [0, 0, int(i),0,0,0,0]
        row_3 += [0, 0, 0,int(i),0,0,0]
        row_4 += [0, 0, 0,0,int(i),0,0]
        row_5 += [0, 0, 0,0,0,int(i),0]
        row_6 += [0, 0, 0,0,0,0,int(i)]
    H_full.append(row_0)
    H_full.append(row_1)
    H_full.append(row_2)
    H_full.append(row_3)
    H_full.append(row_4)
    H_full.append(row_5)
    H_full.append(row_6)



Hx = (H_full @ G_LDPC) % 2



bp_osd = BpOsdDecoder(
            Hx,
            error_rate = 0.1,
            bp_method = 'product_sum',
            max_iter = 30,
            schedule = 'serial',
            osd_method = 'osd_cs', #set to OSD_0 for fast solve
            osd_order = 5
        )

for i in range(100):

    error = np.zeros(208)
    place=np.random.randint(0,208)
    error[place]=1
    syndrome = Hx @ error
    decoding = bp_osd.decode(syndrome)

    if np.array_equal(decoding, error):
        pass
    else:
        print("Failure")
        print(syndrome, place,code_supports[place])



In [135]:
def make_decoder(H):
    return BpOsdDecoder(
        np.array(H),
        error_rate = 0.1,
        bp_method = 'product_sum',
        max_iter = 30,
        schedule = 'serial',
        osd_method = 'osd_cs',
        osd_order = 5
    )

In [150]:
Hx= (H_full @ G_LDPC) % 2


error = np.zeros(208)
place=np.random.randint(0,208)
error[place]=1
syndrome = Hx @ error

def informer_decoder(syndrome, Hx):
    code_0=[]
    code_1=[]
    code_2=[]
    code_3=[]
    code_4=[]
    code_5=[]
    code_6=[]
    s0=[]
    s1=[]
    s2=[]
    s3=[]
    s4=[]
    s5=[]
    s6=[]
    for i in range(3):
        code_0.append(Hx[0+7*i])
        s0.append(syndrome[0+7*i])
        code_1.append(Hx[1+7*i])
        s1.append(syndrome[1+7*i])
        code_2.append(Hx[2+7*i])
        s2.append(syndrome[2+7*i])
        code_3.append(Hx[3+7*i])
        s3.append(syndrome[3+7*i])
        code_4.append(Hx[4+7*i])
        s4.append(syndrome[4+7*i])
        code_5.append(Hx[5+7*i])
        s5.append(syndrome[5+7*i])
        code_6.append(Hx[6+7*i])
        s6.append(syndrome[6+7*i])
    
    dec_0=make_decoder(code_0)
    dec_1=make_decoder(code_1)
    dec_2=make_decoder(code_2)
    dec_3=make_decoder(code_3)
    dec_4=make_decoder(code_4)
    dec_5=make_decoder(code_5)
    dec_6=make_decoder(code_6)

    decoding_0 = dec_0.decode(np.array(s0))
    decoding_1 = dec_1.decode(np.array(s1))
    decoding_2 = dec_2.decode(np.array(s2))
    decoding_3 = dec_3.decode(np.array(s3))
    decoding_4 = dec_4.decode(np.array(s4))
    decoding_5 = dec_5.decode(np.array(s5))
    decoding_6 = dec_6.decode(np.array(s6))
    
    print(Hx@decoding_0)
    print(Hx@decoding_1)
    print(syndrome)

    return(code_0)
tester = informer_decoder(syndrome,Hx)

print(tester)

[1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
[1. 0. 1. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
[array([1, 1, 0, 1, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 1, 0, 1, 1, 0,
       0, 0, 0, 0, 0, 0, 1, 1, 0, 1, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 1, 1,
       0, 1, 0, 1, 1, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 1,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 1, 1,
       1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0,
       0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 1, 0, 0, 0, 0, 0, 0]), array([0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 1, 0, 1, 1, 1, 1, 0, 1, 0, 1, 1, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0

In [ ]:
import ldpc.codes
import ldpc.code_util


In [154]:
d, number_code_words_sampled, lowest_weight_codewords = ldpc.code_util.estimate_code_distance(np.array(tester), timeout_seconds = 15)
print(f"Code distance estimate, d <= {d} (no. codewords sampled: {number_code_words_sampled})")

Code distance estimate, d <= 1 (no. codewords sampled: 6719329)


In [193]:
def spacer(err,i):  #err is small error vectoe from H_outer, i is what code it is, converts small error to error over G_LDPC
    out=np.zeros(49)
    for j in range(7):
        out[j*7+i]= err[j]
    return(np.array(out,dtype=int))

print(spacer([0,0,0,0,0,1,0,0], 0))

[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0
 0 0 0 0 0 0 0 0 0 0 0 0]


[0 0 0 0 0 0 0 1 1 1 1 1 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0]


This decoder works!!!


In [215]:
def peters_decoder(Hx,syndrome):
    s0=[]
    s1=[]
    s2=[]
    s3=[]
    s4=[]
    s5=[]
    s6=[]
    for i in range(3):
        s0.append(syndrome[0+7*i])
        s1.append(syndrome[1+7*i])
        s2.append(syndrome[2+7*i])
        s3.append(syndrome[3+7*i])
        s4.append(syndrome[4+7*i])
        s5.append(syndrome[5+7*i])
        s6.append(syndrome[6+7*i])

    decoder=make_decoder(H_outer)

    dec_0=decoder.decode(np.array(s0))
    dec_1=decoder.decode(np.array(s1))
    dec_2=decoder.decode(np.array(s2))
    dec_3=decoder.decode(np.array(s3))
    dec_4=decoder.decode(np.array(s4))
    dec_5=decoder.decode(np.array(s5))
    dec_6=decoder.decode(np.array(s6))

    

    correc_0=spacer(dec_0,0)
    correc_1=spacer(dec_1,1)
    correc_2=spacer(dec_2,2)
    correc_3=spacer(dec_3,3)
    correc_4=spacer(dec_4,4)
    correc_5=spacer(dec_5,5)
    correc_6=spacer(dec_6,6)

    
    final_correction = correc_0 | correc_1 | correc_2 | correc_3 | correc_4 | correc_5 | correc_6 

    

    recovery = [0]*(208-49) + final_correction.tolist()

    return(np.array(recovery))

    
for d in range(10000):
    error = np.zeros(208)
    place=np.random.randint(0,208)
    error[place]=1


    syndrome = Hx @ error

    correction = peters_decoder(Hx,syndrome)

    if np.array_equal(Hx@correction, syndrome):
            pass
    else:
        print("Failure")

    net_error = (error + correction) % 2

    # syndrome check (optional)
    syndrome = (Hx @ net_error) % 2
    if np.any(syndrome):
        print("Warning: corrected state still has non-zero syndrome!")

    # logical operator check
    logical_effect = (G_LDPC @ net_error) % 2

    if np.any(logical_effect):
        print("❌ Decoder failed: logical operators flipped")
    else:
        pass

    

In [208]:
error = np.zeros(208)
place=np.random.randint(0,208)
error[place]=1


syndrome = Hx @ error

correction = peters_decoder(Hx,syndrome)

print(error)
print(correction)
print(error+correction)
print(Hx@(error+correction)%2)

[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0

✅ Decoder successful: logical operators preserved
